### __??? ????__

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as datetime
import warnings
plt.style.use('fivethirtyeight')
warnings.filterwarnings('ignore')

In [ ]:
import pickle

with open('train_test_data.pkl', 'rb') as f:
    train, test = pickle.load(f)

submission = pd.read_csv('../data/sample_submission.csv', encoding='cp949')

In [ ]:
train.head()

In [ ]:
test.head()

### __?? ??__

In [ ]:
x_train = train.copy()
x_train = x_train.drop(['date_time','power','hour_by_3'], axis = 1, inplace = False)
y_train = train['power'].copy()

x_test = test.copy()
x_test = x_test.drop(['date_time','hour_by_3'], axis = 1, inplace = False)

In [ ]:
x_train.head()

### __????__

??? ????? ????, SMAPE(?? ?? ?? ??? ??)? ????? ???????.
SMAPE? 0?? 200??? ?? ??, 0? ????? ??? ? ? ????.

$$
SMAPE = \frac{1}{n} \sum_{i=1}^{n} \frac{|y_i - \hat{y}_i|}{\frac{|y_i| + |\hat{y}_i|}{2}} \times 100 
$$


SMAPE? ???? ???? ???? ??? ???? ?????, ???? ??? ?? ???? ???? ??? ??? ? ????. ?? ??? ???? ??? ?? ???? ??, ? ????? ?? ??? ??? ??? ?? ??? ??? ????? ????? ????? ?????.

??, SMAPE? ???? ????? ? ?? ?? ? ??? ???? ??? ????, ?? ??? ?? ??? ???? ????. ?? ??? ???? ???? ???? ????? ??? ? ?? ???, ???? ?? ??? ?? ??? ???? ????? ??? ???.

??? ??? SMAPE? ??? ?? ???? ??? ??? ???? ????? ??? ? ?? ????? ?????. ?? ?? ?? ??? ???? ????, ?? ??? ?? ?? ??? ??? ? ????.

In [ ]:
def SMAPE(y_true, y_pred):
    denominator = np.abs(y_true) + np.abs(y_pred)
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0  # ??? 0? ??? ??
    return 200 * np.mean(diff)

### __?? ?? ? ??__

?? KFold? ??? ??, ?? ????? ???? 0?? 46???? ??? ????, ?? ????? 47?? 60???? ??? ???? ?? ?? ??? ??? ??? ? ????. ?? ???? ??, __????? ???? ????? ??__ ??, ???? ???? __????? ???? ????? Stratified K-Fold ???? ??? ??__ ?????.

??? ??? ???? ??? ???????.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold, StratifiedKFold, GridSearchCV

In [ ]:
def num_sfkfold_split(clf, x, y, n_split=5, smape = True):
    rmse_scores = []
    smape_scores = []
    
    sfk = StratifiedKFold(n_split)
  
  
    for train_index, test_index in sfk.split(x, x['num']):
        x_train, x_test = x.values[train_index], x.values[test_index]
        y_train, y_test = y.values[train_index], y.values[test_index]
        
        clf.fit(x_train,y_train)
        pred = clf.predict(x_test)

   
        mse = mean_squared_error(y_test, pred)
        rmse_score = np.sqrt(mse)
        rmse_scores.append(rmse_score)


        if smape == True:
          smape_score = SMAPE(y_test, pred)
          smape_scores.append(smape_score)

    print('???? RMSE: ', rmse_scores)
    print('?? RMSE: ', np.mean(rmse_scores))
  

    if smape == True:
        print('???? SMAPE: ', smape_scores)
        print('?? SMAPE: ', np.mean(smape_scores))

#### __LGBM__

In [ ]:
lgbm = LGBMRegressor(n_estimators = 1000, max_depth = 30, min_child_samples = 100, learning_rate = 0.1, subsample = 0.8, random_state=10, verbose=0)
num_sfkfold_split(lgbm, x_train, y_train, n_split = 5)

#### __Grid Search__

???? ??? ?? ???? ?? XGBoost??? ????? ??? ?? ?? ?????? ???????.

???, ?? ?? ??? ????? ??? ????? ??, ?????? ??????, ?? ????? ???????.

In [ ]:
# param_grid = {
#     'num_leaves': [31, 50, 100],  # ??? ??? ?
#     'learning_rate': [0.01, 0.1, 0.3],  # ???
#     'n_estimators': [100],  # ??? ??
#     'subsample': [0.7, 0.8, 1.0],  # ??? ??? ??
#     'colsample_bytree': [0.7, 0.8, 1.0],  # ?? ??? ??
#     'min_child_samples': [5, 10, 20]  # ?? ??? ?? ?? ?
# }

# lgbm = LGBMRegressor(random_state = 10, verbose = -1)

# grid_search = GridSearchCV(
#     estimator = lgbm,
#     param_grid = param_grid,
#     cv = 5,
#     scoring='neg_mean_squared_error',
#     n_jobs=-1,  
#     verbose=2,  
#     refit=True
# )
# grid_search.fit(x_train, y_train)

# best_params = grid_search.best_params_
# best_model = grid_search.best_estimator_

# with open('best_params.pkl', 'wb') as f:
#     pickle.dump(best_params, f)

# with open('best_model.pkl', 'wb') as f:
#     pickle.dump( grid_search.best_estimator_, f)

# with open('best_model.pkl', 'rb') as f:
#     best_model = pickle.load(f)

# log_sfkfold_split(best_model, x_train, y_train, n_split=5, smape=True)

### __??__

In [ ]:
lgbm.fit(x_train, y_train)

y_pred = lgbm.predict(x_test)

submission['answer'] = y_pred

submission.to_csv('submission.csv', index=False)